In [1]:
import cobra
import pandas as pd 

import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
import gene_information as gi

load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


In [ ]:
def build_expression_reactions(gene_information):
    '''Input is an object of the gene information class. Output is a list of expression reactions for that object'''
    
    

In [4]:
psim_me = pd.read_csv(local_data_path + 'processed/psim_me.csv', index_col = 0)
human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_recon2_2.json')
sp_dict = {1: True, 0: False}
ptm_cols = ['DSB', 'GPI', 'NG', 'OG']
ptm_keys = list(gi.allowed_ptms.keys())

# gene catalyzing metabolic reaction, not processed via secretory pathway
gene1_id = human_model.genes[1].id
# get information from the ME psim that was built (can do user provided information instead)
idx  = psim_me[psim_me['HGNC ID'] == gene1_id].index
ptms_ = dict(zip(ptm_keys, psim_me.loc[idx, ptm_cols].iloc[0,:].tolist()))
ptms_ = {k:v for k,v in ptms_.items() if v != 0}
ptms_['Phosphorylation'] = 3 # example of one that will not be considered
psim_me.Location = psim_me.Location.replace(float('nan'), '0')
fl = [i.replace('[', '') for i in psim_me.loc[idx, 'Location'].tolist()]
fl = [i.replace(']', '') for i in fl]
ensg_ = psim_me.loc[idx, 'Ensembl gene ID'].tolist()[0]

# initialize the gene class
gene1 = gi.gene_information(human_model, hgnc_id = gene1_id, ptms = ptms_, 
                         tmd = psim_me.loc[idx,'TMD'].tolist()[0], sp = sp_dict[psim_me.loc[idx,'SP'].tolist()[0]], 
                           keff = None)
gene1.get_final_locations(metabolic_model = human_model, final_locations=fl)
gene1.get_sequences(ensg_id = ensg_)
gene1.check_gene_information()
print(gene1.module)
print(gene1.hgnc_id)
print(gene1.sp)
print(gene1.ptms)
print(gene1.tmd)
print(gene1.final_locations)


In [5]:
gene1.premrna_seq

'...'